# 04 Geospatial Outage Mapping

## Project: PG&E-Style Utility Operations & Meter-to-Cash Analytics

This notebook prepares the geospatial outage mapping layer for the project. The goal is to load the PG&E outage area GeoJSON file, inspect its structure, clean useful attributes, and export mapping-ready files for later Power BI dashboard development.

The earlier notebooks created the real-data foundation, synthetic Meter-to-Cash model, and SQL reporting layer. This notebook handles the project’s remaining geospatial data source so outage areas can be represented visually in the final dashboard.

In this notebook, I will:

1. Load the raw PG&E outage area GeoJSON file.
2. Inspect the GeoJSON structure and available properties.
3. Validate geometry and attribute fields.
4. Create a cleaned mapping-ready GeoJSON output.
5. Create a summary table for outage area mapping.
6. Export geospatial outputs for later dashboard development.

The purpose of this notebook is to make the PG&E outage area data usable for mapping without overcomplicating the project with a full GIS workflow.

## 1. Import Libraries and Define Project Paths

This section imports the required libraries and defines the raw and geospatial output directories. The GeoJSON file is loaded from the raw data folder and cleaned outputs will be saved to a geospatial folder.

In [3]:
import json
import pandas as pd
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

# Define paths
RAW_DIR = Path("../raw")
PROCESSED_DIR = Path("../data/processed")
GEOSPATIAL_DIR = Path("../data/geospatial")

# Create geospatial output directory
GEOSPATIAL_DIR.mkdir(parents=True, exist_ok=True)

# Define GeoJSON path
GEOJSON_PATH = RAW_DIR / "power_outage_areas.geojson"

# Confirm file exists
print("GeoJSON exists:", GEOJSON_PATH.exists())
print("GeoJSON path:", GEOJSON_PATH)

GeoJSON exists: True
GeoJSON path: ..\raw\power_outage_areas.geojson


In [4]:
# List files in the raw data folder
raw_files = sorted([file.name for file in RAW_DIR.iterdir()])
raw_files

['cec_electricity_county_monthly.csv',
 'cec_electricity_utility_monthly.csv',
 'power_outage_areas.geojson',
 'power_outages_by_county.csv',
 'power_outages_by_incidents.csv']

In [5]:
# Set GeoJSON path using the exact filename shown in the raw folder
GEOJSON_PATH = RAW_DIR / "power_outage_areas.geojson"

print("GeoJSON exists:", GEOJSON_PATH.exists())
print("GeoJSON path:", GEOJSON_PATH)

GeoJSON exists: True
GeoJSON path: ..\raw\power_outage_areas.geojson


In [6]:
# Find any GeoJSON-like files in raw folder
geojson_candidates = [file for file in RAW_DIR.iterdir() if "geojson" in file.name.lower()]
geojson_candidates

[WindowsPath('../raw/power_outage_areas.geojson')]

### GeoJSON File Path Validation

The raw GeoJSON file was found successfully in the `data/raw` folder. This confirms that the notebook can access the outage area mapping file and proceed with structure inspection.

## 2. Load and Inspect GeoJSON Structure

This section loads the outage area GeoJSON file and inspects its top-level structure. The goal is to confirm that the file contains valid GeoJSON features and to identify the available properties for mapping and summary outputs.

In [7]:
# Load GeoJSON file
with open(GEOJSON_PATH, "r", encoding="utf-8") as file:
    outage_geojson = json.load(file)

# Inspect top-level GeoJSON structure
print("GeoJSON type:", outage_geojson.get("type"))
print("Top-level keys:", list(outage_geojson.keys()))

features = outage_geojson.get("features", [])
print("Number of features:", len(features))

# Inspect first feature structure
if features:
    first_feature = features[0]
    print("First feature keys:", list(first_feature.keys()))
    print("First feature geometry type:", first_feature.get("geometry", {}).get("type"))
    print("First feature property keys:", list(first_feature.get("properties", {}).keys()))

GeoJSON type: FeatureCollection
Top-level keys: ['type', 'crs', 'features']
Number of features: 173
First feature keys: ['type', 'id', 'geometry', 'properties']
First feature geometry type: Polygon
First feature property keys: ['OBJECTID', 'UtilityCompany', 'StartDate', 'EstimatedRestoreDate', 'Cause', 'ImpactedCustomers', 'County', 'OutageStatus', 'OutageType', 'IncidentId']


### GeoJSON Structure Findings

The outage area GeoJSON loaded successfully as a valid `FeatureCollection`.

Key observations:

- The file contains 173 geospatial features.
- Each feature uses polygon geometry, which is appropriate for mapping outage areas.
- The available properties include utility company, start date, estimated restoration date, cause, impacted customers, county, outage status, outage type, and incident ID.
- These fields align closely with the outage incident data used earlier in the project, but this file adds polygon geometry for mapping.

This confirms that the GeoJSON can be used as the geospatial outage area layer for later dashboard mapping.

## 3. Convert GeoJSON Properties to a DataFrame

This section extracts the GeoJSON feature properties into a tabular dataframe. The geometry is kept inside the original GeoJSON file, while the properties dataframe is used for profiling, cleaning, and summary outputs.

In [8]:
# Extract feature properties into a dataframe
geo_properties = pd.DataFrame([
    feature.get("properties", {}) for feature in features
])

print("GeoJSON properties dataframe:", geo_properties.shape)
geo_properties.head()

GeoJSON properties dataframe: (173, 10)


,OBJECTID,UtilityCompany,StartDate,EstimatedRestoreDate,Cause,ImpactedCustomers,County,OutageStatus,OutageType,IncidentId
0,10197758,PGE,"Mon, 08 Jun 2026 20:35:00 GMT","Wed, 10 Jun 2026 22:00:00 GMT",POLE FIRE,25,SOLANO,Active,Not Planned,274224
1,10197759,PGE,"Mon, 08 Jun 2026 20:35:00 GMT","Wed, 10 Jun 2026 22:00:00 GMT",POLE FIRE,7,SOLANO,Active,Not Planned,274235
2,10197760,PGE,"Tue, 09 Jun 2026 11:06:29 GMT","Tue, 09 Jun 2026 12:00:00 GMT",PLNND SHUTDOWN,1,SAN MATEO,Active,Planned,274764
3,10197761,PGE,"Tue, 09 Jun 2026 15:31:00 GMT","Tue, 09 Jun 2026 23:30:00 GMT",PLNND SHUTDOWN,6,EL DORADO,Active,Planned,275014
4,10197762,PGE,"Tue, 09 Jun 2026 16:13:49 GMT","Tue, 09 Jun 2026 19:30:00 GMT",PLNND SHUTDOWN,32,SAN MATEO,Active,Planned,275057


### GeoJSON Properties Findings

The GeoJSON feature properties were successfully extracted into a dataframe with 173 records and 10 fields.

Key observations:

- The properties include utility company, outage start date, estimated restoration date, cause, impacted customers, county, outage status, outage type, and incident ID.
- The records shown are PG&E outage area records, which aligns with the purpose of this notebook.
- These attributes can be used to create a mapping summary table while the original GeoJSON geometry is preserved for polygon mapping.
- The GeoJSON dates are stored as text and will need to be converted into datetime fields before export.

## 4. Clean GeoJSON Attribute Fields

This section standardizes the GeoJSON property fields. Column names are converted to snake_case, text fields are cleaned, impacted customers are converted to numeric values, and outage dates are converted into datetime fields.

In [9]:
# Create cleaned copy of GeoJSON properties
geo_properties_clean = geo_properties.copy()

# Rename columns to snake_case
geo_properties_clean = geo_properties_clean.rename(columns={
    "OBJECTID": "object_id",
    "UtilityCompany": "utility_company",
    "StartDate": "start_datetime",
    "EstimatedRestoreDate": "estimated_restoration_datetime",
    "Cause": "cause",
    "ImpactedCustomers": "impacted_customers",
    "County": "county",
    "OutageStatus": "outage_status",
    "OutageType": "outage_type",
    "IncidentId": "incident_id"
})

# Standardize text fields
geo_properties_clean["utility_company"] = geo_properties_clean["utility_company"].str.upper().str.strip()
geo_properties_clean["county"] = geo_properties_clean["county"].str.upper().str.strip()
geo_properties_clean["outage_status"] = geo_properties_clean["outage_status"].str.title().str.strip()
geo_properties_clean["outage_type"] = geo_properties_clean["outage_type"].str.title().str.strip()
geo_properties_clean["cause"] = geo_properties_clean["cause"].fillna("UNKNOWN / NOT PROVIDED").str.upper().str.strip()

# Convert numeric field
geo_properties_clean["impacted_customers"] = pd.to_numeric(
    geo_properties_clean["impacted_customers"],
    errors="coerce"
)

# Convert datetime fields
geo_properties_clean["start_datetime"] = pd.to_datetime(
    geo_properties_clean["start_datetime"],
    errors="coerce",
    utc=True
)

geo_properties_clean["estimated_restoration_datetime"] = pd.to_datetime(
    geo_properties_clean["estimated_restoration_datetime"],
    errors="coerce",
    utc=True
)

# Create restoration duration in hours
geo_properties_clean["estimated_restoration_hours"] = (
    geo_properties_clean["estimated_restoration_datetime"]
    - geo_properties_clean["start_datetime"]
).dt.total_seconds() / 3600

print("Clean GeoJSON properties:", geo_properties_clean.shape)
geo_properties_clean.head()

Clean GeoJSON properties: (173, 11)


,object_id,utility_company,start_datetime,estimated_restoration_datetime,cause,impacted_customers,county,outage_status,outage_type,incident_id,estimated_restoration_hours
0,10197758,PGE,2026-06-08 20:35:00+00:00,2026-06-10 22:00:00+00:00,POLE FIRE,25,SOLANO,Active,Not Planned,274224,49.416667
1,10197759,PGE,2026-06-08 20:35:00+00:00,2026-06-10 22:00:00+00:00,POLE FIRE,7,SOLANO,Active,Not Planned,274235,49.416667
2,10197760,PGE,2026-06-09 11:06:29+00:00,2026-06-09 12:00:00+00:00,PLNND SHUTDOWN,1,SAN MATEO,Active,Planned,274764,0.891944
3,10197761,PGE,2026-06-09 15:31:00+00:00,2026-06-09 23:30:00+00:00,PLNND SHUTDOWN,6,EL DORADO,Active,Planned,275014,7.983333
4,10197762,PGE,2026-06-09 16:13:49+00:00,2026-06-09 19:30:00+00:00,PLNND SHUTDOWN,32,SAN MATEO,Active,Planned,275057,3.269722


### GeoJSON Attribute Cleaning Findings

The GeoJSON outage area attributes were cleaned successfully.

Key observations:

- Column names were standardized into snake_case format.
- Utility company, county, outage status, outage type, and cause fields were cleaned for consistency.
- Impacted customers were converted into a numeric field.
- Start and estimated restoration timestamps were converted into datetime fields.
- Estimated restoration duration was calculated in hours.
- The cleaned attribute table now has 173 records and 11 fields.

This cleaned attribute layer can be used for outage area summaries while the original GeoJSON geometry remains available for polygon mapping.

## 5. Profile Cleaned GeoJSON Attributes

This section reviews data types, missing values, utility companies, outage types, and county coverage in the cleaned GeoJSON attribute table.

In [10]:
# Profile cleaned GeoJSON attributes
geo_profile = pd.DataFrame({
    "dtype": geo_properties_clean.dtypes,
    "missing_count": geo_properties_clean.isna().sum(),
    "missing_pct": (geo_properties_clean.isna().mean() * 100).round(2),
    "unique_values": geo_properties_clean.nunique()
})

display(geo_profile)

print("Utility companies:")
print(sorted(geo_properties_clean["utility_company"].dropna().unique()))

print("\nOutage types:")
print(sorted(geo_properties_clean["outage_type"].dropna().unique()))

print("\nOutage statuses:")
print(sorted(geo_properties_clean["outage_status"].dropna().unique()))

print("\nCounty count:", geo_properties_clean["county"].nunique())

,dtype,missing_count,missing_pct,unique_values
object_id,int64,0,0.00,173
utility_company,object,0,0.00,1
start_datetime,"datetime64[ns, UTC]",0,0.00,157
estimated_restoration_datetime,"datetime64[ns, UTC]",12,6.94,41
cause,object,0,0.00,12
impacted_customers,int64,0,0.00,46
county,object,0,0.00,33
outage_status,object,0,0.00,1
outage_type,object,0,0.00,2
incident_id,object,0,0.00,173


Utility companies:
['PGE']

Outage types:
['Not Planned', 'Planned']

Outage statuses:
['Active']

County count: 33


### Cleaned GeoJSON Attribute Profile Findings

The cleaned GeoJSON attribute profile confirms that the outage area file is usable for PG&E outage mapping.

Key observations:

- The file contains 173 PG&E outage area polygons.
- All records belong to `PGE`, which makes this file directly relevant to the project.
- The outage areas cover 33 counties.
- All records are marked as active in the current outage snapshot.
- Outage type includes both planned and not planned outage areas.
- Estimated restoration timestamps are missing for 12 records, which also creates missing estimated restoration duration values.
- Impacted customer values are complete and stored as numeric data.
- Each outage area has a unique incident ID.

This confirms that the GeoJSON can support a mapping layer showing current PG&E outage areas, impacted customers, outage type, county, and estimated restoration timing.

## 6. Create GeoJSON County Summary

This section summarizes the cleaned outage area attributes by county. The goal is to create a mapping-friendly county summary that can be used alongside the polygon GeoJSON layer.

In [11]:
# Summarize GeoJSON outage areas by county
geo_county_summary = (
    geo_properties_clean
    .groupby("county", as_index=False)
    .agg(
        outage_area_count=("incident_id", "nunique"),
        total_impacted_customers=("impacted_customers", "sum"),
        avg_impacted_customers=("impacted_customers", "mean"),
        median_impacted_customers=("impacted_customers", "median"),
        planned_outage_areas=("outage_type", lambda x: (x == "Planned").sum()),
        not_planned_outage_areas=("outage_type", lambda x: (x == "Not Planned").sum()),
        avg_estimated_restoration_hours=("estimated_restoration_hours", "mean"),
        max_estimated_restoration_hours=("estimated_restoration_hours", "max")
    )
    .sort_values("total_impacted_customers", ascending=False)
)

# Round numeric columns for readability
for col in [
    "avg_impacted_customers",
    "median_impacted_customers",
    "avg_estimated_restoration_hours",
    "max_estimated_restoration_hours"
]:
    geo_county_summary[col] = geo_county_summary[col].round(2)

geo_county_summary.head(15)

,county,outage_area_count,total_impacted_customers,avg_impacted_customers,median_impacted_customers,planned_outage_areas,not_planned_outage_areas,avg_estimated_restoration_hours,max_estimated_restoration_hours
4,CONTRA COSTA,6,4141,690.17,18.0,4,2,6.43,7.88
24,SANTA CLARA,18,1767,98.17,6.5,7,11,8.56,29.90
0,ALAMEDA,17,478,28.12,7.0,5,12,8.58,24.25
22,SAN MATEO,15,433,28.87,11.0,9,6,6.23,10.05
10,MARIN,2,326,163.00,163.0,1,1,8.60,10.22
17,PLACER,5,243,48.60,3.0,3,2,6.40,7.92
19,SAN FRANCISCO,7,101,14.43,1.0,1,6,5.57,7.36
16,NEVADA,5,91,18.20,4.0,4,1,5.65,7.97
21,SAN LUIS OBISPO,9,84,9.33,2.0,6,3,6.32,7.99
5,EL DORADO,9,67,7.44,3.0,7,2,5.96,7.98


### GeoJSON County Summary Findings

The GeoJSON county summary shows where PG&E outage area polygons are concentrated and where customer impact is highest.

Key observations:

- Contra Costa has the highest total impacted customer count in the outage area layer.
- Santa Clara and Alameda have the highest outage area counts, indicating more mapped outage polygons.
- San Mateo also has a relatively high number of outage areas with both planned and not-planned outage types.
- Contra Costa has a much higher average impacted customer count than most counties, suggesting that a small number of outage polygons account for a large share of customer impact.
- Planned and not-planned outage area counts can support map filters and outage-type comparisons in Power BI.

This summary provides a county-level companion table for the polygon GeoJSON mapping layer.

## 7. Compare GeoJSON County Summary to Processed Outage Summary

This section compares the GeoJSON outage area county summary with the processed PG&E county outage summary from Notebook 1. The goal is to check whether the polygon outage area layer aligns with the tabular outage incident summary.

In [12]:
# Load processed PG&E county outage summary from Notebook 1
pge_county_outage_summary = pd.read_csv(
    PROCESSED_DIR / "pge_county_outage_summary.csv"
)

# Compare GeoJSON county summary with processed outage summary
geo_vs_processed_county = (
    geo_county_summary[
        [
            "county",
            "outage_area_count",
            "total_impacted_customers",
            "planned_outage_areas",
            "not_planned_outage_areas"
        ]
    ]
    .merge(
        pge_county_outage_summary[
            [
                "county",
                "outage_incidents",
                "total_impacted_customers"
            ]
        ],
        on="county",
        how="outer",
        suffixes=("_geojson", "_processed")
    )
)

# Calculate differences
geo_vs_processed_county["outage_count_difference"] = (
    geo_vs_processed_county["outage_area_count"]
    - geo_vs_processed_county["outage_incidents"]
)

geo_vs_processed_county["impacted_customer_difference"] = (
    geo_vs_processed_county["total_impacted_customers_geojson"]
    - geo_vs_processed_county["total_impacted_customers_processed"]
)

geo_vs_processed_county = geo_vs_processed_county.sort_values(
    "total_impacted_customers_geojson",
    ascending=False
)

geo_vs_processed_county.head(20)

,county,outage_area_count,total_impacted_customers_geojson,planned_outage_areas,not_planned_outage_areas,outage_incidents,total_impacted_customers_processed,outage_count_difference,impacted_customer_difference
5,CONTRA COSTA,6.0,4141.0,4.0,2.0,6,4141,0.0,0.0
28,SANTA CLARA,18.0,1767.0,7.0,11.0,21,933,-3.0,834.0
0,ALAMEDA,17.0,478.0,5.0,12.0,15,476,2.0,2.0
26,SAN MATEO,15.0,433.0,9.0,6.0,16,434,-1.0,-1.0
14,MARIN,2.0,326.0,1.0,1.0,3,327,-1.0,-1.0
21,PLACER,5.0,243.0,3.0,2.0,2,202,3.0,41.0
23,SAN FRANCISCO,7.0,101.0,1.0,6.0,6,100,1.0,1.0
20,NEVADA,5.0,91.0,4.0,1.0,3,84,2.0,7.0
25,SAN LUIS OBISPO,9.0,84.0,6.0,3.0,8,83,1.0,1.0
6,EL DORADO,9.0,67.0,7.0,2.0,9,75,0.0,-8.0


### GeoJSON vs Processed Outage Summary Findings

The GeoJSON county summary was compared with the processed PG&E county outage summary from Notebook 1.

Key observations:

- The major impacted counties align across both sources, including Contra Costa, Santa Clara, Alameda, San Mateo, Marin, and Placer.
- Contra Costa remains the highest-impact county in both the polygon outage area layer and the processed outage summary.
- Some outage counts differ because the GeoJSON represents outage area polygons, while the processed outage summary was built from the tabular incident layer.
- The polygon layer is best suited for mapping outage areas, while the processed outage summary is better suited for tabular operational reporting.
- Differences between outage area counts and incident counts should be documented rather than treated as errors.

This comparison confirms that the GeoJSON layer is directionally consistent with the earlier processed outage analysis and can be used as a mapping layer in the final dashboard.

## 8. Export Geospatial Mapping Outputs

This section exports the cleaned GeoJSON and summary tables for later Power BI dashboard development. The cleaned GeoJSON preserves polygon geometry, while the CSV summaries provide county-level mapping context.

In [13]:
# Create a cleaned GeoJSON copy with standardized properties
cleaned_geojson = outage_geojson.copy()

for feature, (_, row) in zip(cleaned_geojson["features"], geo_properties_clean.iterrows()):
    feature["properties"] = {
        "object_id": int(row["object_id"]),
        "utility_company": row["utility_company"],
        "start_datetime": row["start_datetime"].isoformat() if pd.notna(row["start_datetime"]) else None,
        "estimated_restoration_datetime": row["estimated_restoration_datetime"].isoformat() if pd.notna(row["estimated_restoration_datetime"]) else None,
        "cause": row["cause"],
        "impacted_customers": int(row["impacted_customers"]) if pd.notna(row["impacted_customers"]) else None,
        "county": row["county"],
        "outage_status": row["outage_status"],
        "outage_type": row["outage_type"],
        "incident_id": row["incident_id"],
        "estimated_restoration_hours": round(row["estimated_restoration_hours"], 2) if pd.notna(row["estimated_restoration_hours"]) else None
    }

# Define export paths
clean_geojson_path = GEOSPATIAL_DIR / "pge_outage_areas_clean.geojson"
geo_properties_path = GEOSPATIAL_DIR / "pge_outage_area_properties_clean.csv"
geo_county_summary_path = GEOSPATIAL_DIR / "pge_outage_area_county_summary.csv"
geo_comparison_path = GEOSPATIAL_DIR / "pge_geojson_vs_processed_county_comparison.csv"

# Export cleaned GeoJSON
with open(clean_geojson_path, "w", encoding="utf-8") as file:
    json.dump(cleaned_geojson, file)

# Export CSV summaries
geo_properties_clean.to_csv(geo_properties_path, index=False)
geo_county_summary.to_csv(geo_county_summary_path, index=False)
geo_vs_processed_county.to_csv(geo_comparison_path, index=False)

# Confirm exported files
geospatial_files = sorted([file.name for file in GEOSPATIAL_DIR.iterdir()])
geospatial_files

['pge_geojson_vs_processed_county_comparison.csv',
 'pge_outage_area_county_summary.csv',
 'pge_outage_area_properties_clean.csv',
 'pge_outage_areas_clean.geojson']

### Geospatial Export Summary

The geospatial mapping outputs were exported successfully.

The exported files include:

- `pge_outage_areas_clean.geojson`: cleaned polygon GeoJSON layer for PG&E outage area mapping.
- `pge_outage_area_properties_clean.csv`: cleaned outage area attribute table.
- `pge_outage_area_county_summary.csv`: county-level summary of outage area counts, impacted customers, planned outage areas, and not-planned outage areas.
- `pge_geojson_vs_processed_county_comparison.csv`: comparison between the GeoJSON outage area layer and the processed outage summary from Notebook 1.

These files can be used later for Power BI mapping, geospatial review, and outage context reporting.

## 9. Notebook Summary and Next Steps

This notebook prepared the geospatial outage mapping layer for the PG&E-style Utility Operations & Meter-to-Cash Analytics project.

Key outcomes:

- Loaded the raw PG&E outage area GeoJSON file.
- Confirmed that the file is a valid GeoJSON `FeatureCollection`.
- Identified 173 polygon outage area features.
- Extracted GeoJSON feature properties into a dataframe.
- Standardized column names and cleaned utility, county, outage status, outage type, and cause fields.
- Converted impacted customers to numeric values.
- Converted outage start and estimated restoration timestamps into datetime fields.
- Calculated estimated restoration duration in hours.
- Profiled the cleaned GeoJSON attributes and confirmed that all records belong to PGE.
- Created a county-level outage area summary for mapping and dashboard context.
- Compared the GeoJSON county summary against the processed PG&E outage summary from Notebook 1.
- Exported cleaned geospatial outputs to `data/geospatial/`.

The next phase of the project will focus on Power BI dashboard preparation and design. The reporting CSV files from Notebook 3 and geospatial outputs from this notebook will be used to build dashboard pages covering executive KPIs, billing performance, billing exceptions, service request backlog, account readiness, and PG&E outage/consumption context.